# 질감 모델은 패치의 어느 위치에 반응할까?

저장된 질감 가림 모델로 같은 프레임에 같은 바닥 질감을 붙여 비교합니다. 앞선 가림 실험에서 빈 바닥까지 검수한 5명 31장을 사용합니다. 각 프레임의 패치는 다른 검증 수행자의 빈 바닥에서 가져옵니다. 학습은 실행하지 않습니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from ultralytics import YOLO

from wlcv.config import DATASET, RUNS
from wlcv.evaluation.patch_position import (
    reviewed_floor_frames, make_example, paste_patch, evaluate_positions, summarize,
)

run = RUNS / "plate_occlusion_texture"
positions = RUNS / "plate_baseline/report/occlusion_half_area/mask_positions.csv"
floors = reviewed_floor_frames(positions)
trained_people = set(pd.read_csv(run / "train_people.csv").person_id)
assert trained_people.isdisjoint(floors.person)
model = YOLO(str((run / "weights/best.pt").resolve(strict=True)))
assert model.names == {0: "plate"}
out = run / "report/patch_position"
out.mkdir(parents=True, exist_ok=True)
display(floors.groupby("person").size().rename("frames"))

## 1. 원본과 합성 위치 먼저 보기

작은 조각(strip)을 중앙·아래 끝·빈 바닥으로 옮깁니다. 윤곽을 넘어가는 조건(cross lower edge)은 아래 끝에 놓은 픽셀을 그대로 두고 같은 바닥 무늬를 박스 밖으로 연장합니다. 이에 대응해 큰 조각(full)을 중앙과 빈 바닥에 붙인 대조도 둡니다. 정답 박스는 실선, 붙인 패치는 점선입니다. 빈 바닥 조건은 바닥 주변을 확대합니다. 위치·크기·정답을 확인한 뒤 다음 셀을 실행하세요.

In [ ]:
example = "a03_sn_f_01_f19"
image, truth, floor, donor, variants = make_example(DATASET, floors, example)
fig, axes = plt.subplots(2, 4, figsize=(12, 6), layout="constrained")
for axis, (condition, placement) in zip(axes.flat, variants.items()):
    axis.imshow(paste_patch(image, placement))
    x1, y1, x2, y2 = truth
    axis.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False, edgecolor="#42d7ed"))
    if placement is not None:
        x, y, patch = placement
        h, w = patch.shape[:2]
        axis.add_patch(Rectangle((x,y), w, h, fill=False, edgecolor="#ffba42", linestyle="--"))
    if condition.startswith("Floor"):
        x1,y1,x2,y2 = floor
    axis.set_xlim(max(0,x1-65), min(image.shape[1],x2+65))
    axis.set_ylim(min(image.shape[0],y2+65), max(0,y1-65))
    axis.set_title(condition)
    axis.axis("off")
axes.flat[-1].axis("off")
fig.suptitle(f"{example}   |   Floor texture from {donor}")
fig.savefig(out / "patch_examples.png", dpi=150)
plt.show()

## 2. 일곱 조건을 같은 기준으로 추론

CPU·입력 크기 640, 후보 신뢰도 하한 0.001입니다. 원본 정답과 IoU ≥ 0.50인 후보 중 최고 신뢰도가 0.10 이상이면 검출입니다. 원본과 합성 프레임의 정답은 동일합니다. 빈 바닥 오탐은 같은 바닥 영역에 중심이 있는 비매칭 박스가 신뢰도 0.10 이상인 프레임 수입니다. `coverage`는 원본 정답 박스를 정수 픽셀로 옮겨 계산한 가림 비율입니다.

`lost`와 `recovered`는 원본 대비 미탐·복구입니다. 신뢰도·IoU 변화의 중앙값은 원본과 해당 조건 양쪽에서 후보가 매칭된 프레임만 사용하고, 그 장수를 `paired_matches`에 적습니다.

In [ ]:
measurements = evaluate_positions(model, DATASET, floors)
paired, summary = summarize(measurements)
paired.to_csv(out / "per_frame.csv", index=False)
summary.to_csv(out / "summary.csv")
display(summary.round(4))
by_person = paired.pivot_table(index="person", columns="condition", values="detected", aggfunc="sum")
display(by_person[summary.index])
by_person[summary.index].to_csv(out / "by_person.csv")

## 3. 검출 수와 실제 사례 비교

결과를 사람별로도 확인하세요. 예시의 IoU와 신뢰도는 정답과 매칭된 후보의 값입니다. 후보가 없으면 빈 값으로 남깁니다.

In [ ]:
order = ["Original", "Center strip", "Lower strip", "Cross lower edge", "Floor strip",
         "Center full", "Floor full"]
bars = summary.loc[order, "detected"]
fig, axis = plt.subplots(figsize=(8, 3.9), layout="constrained")
colors = ["#677681", "#318594", "#318594", "#206473", "#869c89", "#318594", "#869c89"]
artists = axis.barh(order, bars, color=colors)
axis.bar_label(artists, padding=4)
axis.invert_yaxis()
axis.set(xlim=(0,len(floors)+1), xlabel=f"Detected frames out of {len(floors)}",
         title="Texture model · confidence ≥ 0.10 and IoU ≥ 0.50")
fig.savefig(out / "position_comparison.png", dpi=150)
plt.show()
display(paired.loc[paired.stem == example, ["condition", "coverage", "confidence", "iou", "detected"]].round(3))

## 4. 같은 패치에서 베이스라인과 질감 모델 비교

앞선 질감 모델의 `per_frame.csv`가 있으면 커널을 다시 열어도 이 절부터 실행할 수 있습니다. 기준 모델의 가중치만 바꿔 31장 × 7조건을 추론합니다. 대상 프레임, 패치의 출처·좌표·크기·픽셀 해시, 정답 박스에서 가린 면적을 이전 기록과 대조합니다. 신뢰도 0.10, 매칭 IoU 0.50을 유지합니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

from wlcv.config import DATASET, RUNS
from wlcv.evaluation.patch_position import evaluate_positions, reviewed_floor_frames, summarize

baseline_run = RUNS / "plate_baseline"
texture_report = RUNS / "plate_occlusion_texture/report/patch_position"
comparison_dir = baseline_run / "report/patch_position_comparison"
positions = baseline_run / "report/occlusion_half_area/mask_positions.csv"
reviewed = reviewed_floor_frames(positions)
reference = pd.read_csv(texture_report / "per_frame.csv")
trained_people = pd.read_csv(baseline_run / "train_people.csv").person_id
assert set(reviewed.person).isdisjoint(trained_people)
assert len(reviewed) == 31

baseline_model = YOLO(str((baseline_run / "weights/best.pt").resolve(strict=True)))
assert baseline_model.names == {0: "plate"}
baseline_rows = evaluate_positions(baseline_model, DATASET, reviewed, threshold=0.10)
baseline_scores, baseline_summary = summarize(baseline_rows)

# 패치 해시는 실제로 붙인 픽셀값으로 계산했다. 원본 행에는 패치가 없다.
keys = ["stem", "condition"]
inputs = ["person", "frame", "donor", "x", "y", "width", "height", "coverage", "patch_hash"]
base_inputs = baseline_scores.set_index(keys)[inputs].sort_index()
saved_inputs = reference.set_index(keys)[inputs].sort_index()
assert base_inputs.index.is_unique and saved_inputs.index.is_unique
pd.testing.assert_frame_equal(base_inputs, saved_inputs, check_exact=False, rtol=0, atol=1e-12)

both = pd.concat([baseline_scores.assign(model="Baseline"), reference.assign(model="Texture")],
                 ignore_index=True)
_, texture_summary = summarize(reference[baseline_rows.columns])
totals = pd.concat([baseline_summary.assign(model="Baseline"), texture_summary.assign(model="Texture")])

### 중앙에서 아래쪽으로 패치를 옮기면 어떤 프레임이 달라질까?

`lost`는 중앙에서 정탐이었지만 아래쪽에서 미탐인 장수이고, `recovered`는 그 반대입니다. 원본 대비 변화의 중앙값은 양쪽 모두 매칭된 후보가 있는 프레임에서 계산합니다. `paired_matches`가 해당 장수입니다.

In [ ]:
# 같은 프레임에서 중앙 검출을 잃었는지, 중앙 미탐을 되찾았는지 각각 센다.
from_center = []
for name, scores in both.groupby("model", sort=False):
    hits = scores.pivot(index=["person", "stem"], columns="condition", values="detected")
    for condition in ["Lower strip", "Cross lower edge"]:
        paired = pd.DataFrame({
            "center_detected": hits["Center strip"], "edge_detected": hits[condition],
            "lost": hits["Center strip"] & ~hits[condition],
            "recovered": ~hits["Center strip"] & hits[condition],
        })
        from_center.append(paired.reset_index().assign(model=name, condition=condition))
location_changes = pd.concat(from_center, ignore_index=True)
location_counts = location_changes.groupby(["model", "condition"], sort=False)[["lost", "recovered"]].sum()

comparison_dir.mkdir(parents=True, exist_ok=True)
both.to_csv(comparison_dir / "per_frame.csv", index=False)
totals.to_csv(comparison_dir / "summary.csv")
location_changes.to_csv(comparison_dir / "location_changes.csv", index=False)
location_counts.to_csv(comparison_dir / "location_counts.csv")
display(totals[["model", "frames", "detected", "floor_fp", "paired_matches",
                "median_delta_conf", "median_delta_iou"]].round(4))
display(location_counts)

In [ ]:
order = ["Original", "Center strip", "Lower strip", "Cross lower edge",
         "Center full", "Floor strip", "Floor full"]
detected = totals.reset_index().pivot(index="condition", columns="model", values="detected")
fig, axis = plt.subplots(figsize=(8.4, 4.8), layout="constrained")
detected.loc[order, ["Baseline", "Texture"]].plot.barh(ax=axis, color=["#225a78", "#d27c37"], width=0.78)
for bars in axis.containers:
    axis.bar_label(bars, padding=3, fontsize=9)
axis.set(xlabel="Detected frames (out of 31)", ylabel="", xlim=(0, 33),
         title="Plate detection by patch position · confidence ≥ 0.10, IoU ≥ 0.50")
axis.invert_yaxis()
handles, names = axis.get_legend_handles_labels()
axis.get_legend().remove()
fig.legend(handles, names, loc="outside lower center", ncol=2, frameon=False)
fig.savefig(comparison_dir / "position_by_model.png", dpi=150)
plt.show()

### 같은 입력에서 두 모델의 점수 확인

`a03` 19번과 `b02` 30번의 실제 합성 입력입니다. 청록색은 원본 정답 박스입니다. 제목은 해당 정답과 IoU가 0.50 이상인 후보의 최고 신뢰도입니다. 평가 기준은 0.10입니다.

In [ ]:
from matplotlib.patches import Rectangle
from wlcv.evaluation.patch_position import make_example, paste_patch

conditions = ["Center strip", "Lower strip", "Cross lower edge"]
examples = ["a03_sn_f_01_f19", "b02_sn_h_02_f30"]
scores_by_frame = both.set_index(["stem", "condition", "model"])
fig, axes = plt.subplots(2, 3, figsize=(10, 6.7), layout="constrained")
for row, stem in enumerate(examples):
    image, truth, floor, donor, patches = make_example(DATASET, reviewed, stem)
    x1, y1, x2, y2 = truth
    for col, condition in enumerate(conditions):
        axis = axes[row, col]
        axis.imshow(paste_patch(image, patches[condition]))
        axis.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="#52ecf5"))
        margin = (x2-x1) * 0.4
        axis.set_xlim(max(0, x1-margin), min(image.shape[1], x2+margin))
        axis.set_ylim(min(image.shape[0], y2+margin), max(0, y1-margin))
        labels = []
        for model_name in ["Baseline", "Texture"]:
            score = scores_by_frame.loc[(stem, condition, model_name)]
            text = f"{score.confidence:.3f}" if pd.notna(score.confidence) else "no match"
            labels.append(f"{model_name} {text}")
        axis.set_title(f"{stem}  ·  {condition}\n" + "  |  ".join(labels), fontsize=9)
        axis.axis("off")
fig.suptitle("Confidence of a candidate matching the original box (cyan)")
fig.savefig(comparison_dir / "paired_frames.png", dpi=150)
plt.show()